# Day 1 — Build a Local Voice Agent (Sarvam + Groq)

**Module 9 · Voice Agent Testing**

---

## What we'll do today

| # | Step | Why |
|---|---|---|
| 1 | Understand the voice-agent pipeline | STT -> LLM -> TTS, and why **latency** rules everything |
| 2 | Meet the stack: **Sarvam** (STT+TTS) + **Groq** (the brain) | Free, fast, streaming; strong on Indian languages/accents |
| 3 | Build & run a local agent | audio question in -> spoken answer out, every stage **timed** |
| 4 | Introduce **voice-specific testing** | What's genuinely new vs. the LLM testing you already know (Day 2 implements it) |

**Estimated time:** 60 minutes

---

> **Where we are in the course**
> You have spent eight modules testing the *text* brain of AI systems — relevancy, faithfulness, tool calls, adversarial resistance. A voice agent wraps that same brain in **ears** (speech-to-text) and a **mouth** (text-to-speech). The brain testing is *unchanged* (you'll reuse promptfoo/DeepEval). Today is about the two new organs and the one property that dominates voice — **latency** — so that Day 2 you can test them.

## 1. What is a voice agent?

A voice agent is a three-stage pipeline. A person speaks; three components run in series; the person hears a reply:

```
  🎙️ audio in
      │
      ▼
  [ STT ]  speech-to-text   — the "ears"  (Sarvam saarika)
      │  transcript (text)
      ▼
  [ LLM ]  the "brain"      — decides what to say  (Groq, streamed)
      │  reply (text)
      ▼
  [ TTS ]  text-to-speech   — the "mouth"  (Sarvam bulbul)
      │
      ▼
  🔊 audio out
```

Two things make this different from every system you've tested so far:

1. **Three failure surfaces, not one.** The ears can mishear ("Manali" → "monolli"), the brain can answer wrong, *and* the mouth can mispronounce. A perfect brain still fails if the ears mishear the question.
2. **Latency is a feature, not a footnote.** In text, a 3-second wait is fine. In a conversation, 3 seconds of silence feels *broken*. That's why the brain must **stream** (start talking before it's finished thinking) and why our agent times every stage.

> **Plain English:** you've been grading the chef's cooking. A voice agent also has a waiter taking the order by ear in a noisy room, and another reading the dish back aloud. Any of the three can ruin the meal — and if any of them is slow, the diner walks out.

## 2. The stack — Sarvam + Groq

- **Sarvam** ([playground](https://platform.sarvam.ai/api-playground/)) provides the **ears and mouth**:
  - **STT** (`saarika`) — speech → text. Strong on Indian languages, accents, and code-switching (Hindi-English mixing) — the exact conditions generic STT struggles with.
  - **TTS** (`bulbul`) — text → speech, with named speakers (e.g. `anushka`).
  - Official Python SDK: `pip install sarvamai`.
- **Groq** provides the **brain** — an LLM served over an OpenAI-compatible API that is *very fast* and supports streaming. Speed here is what makes the agent feel alive. Default model: `llama-3.1-8b-instant`.

Both have free tiers. Get keys from [platform.sarvam.ai](https://platform.sarvam.ai/) and [console.groq.com](https://console.groq.com/), and put them in `examples/.env` (copy `.env.example`). The cell below checks they're present.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

for key in ("SARVAM_API_KEY", "GROQ_API_KEY"):
    v = os.getenv(key, "")
    print(f"{key:<16}: {'set (' + str(len(v)) + ' chars)' if v and 'your-' not in v else 'MISSING -> add it to examples/.env'}")
print("\nInstall deps first if needed:  pip install -r ../requirements.txt")

SARVAM_API_KEY  : set (36 chars)
GROQ_API_KEY    : set (56 chars)

Install deps first if needed:  pip install -r ../requirements.txt


## 3. Look at the agent

`voice_agent.py` is the whole thing — a `VoiceAgent` with three timed stages (`transcribe` → `think` → `speak`) and a `respond()` that chains them into a `VoiceTurn`. Two design choices are already **testing decisions**:

1. **It's file-in / file-out.** Audio file → `VoiceTurn(transcript, reply_text, audio_out_path, timings_ms)`. Every stage returns a value you can assert on — that's what makes Day 2 possible.
2. **The system prompt forces *speakable* replies** (1–3 short sentences, no markdown/bullets/emojis). A brain that returns a bulleted list is *correct text* but a *broken voice reply*. That's a voice-specific requirement you'll test on Day 2.

In [2]:
# Read the agent (structure over detail)
src = open("voice_agent.py").read()
print(src[src.index("DEFAULT_SYSTEM_PROMPT"):src.index("class VoiceAgent")])   # the speakable-reply prompt
print("... class VoiceAgent: transcribe() -> think() -> speak() -> respond() ...")

DEFAULT_SYSTEM_PROMPT = (
    "You are a friendly voice assistant. Reply in 1-3 short, conversational "
    "sentences that are easy to say out loud. Do not use markdown, bullet points, "
    "code, emojis, or symbols — only plain spoken words. If you don't know, say so."
)


@dataclass
class VoiceTurn:
    """Everything one turn produced — each field is something a test can assert on."""
    transcript: str            # what Sarvam STT heard
    reply_text: str            # what the Groq LLM said (text)
    audio_out_path: str        # where the TTS .wav was written
    timings_ms: dict = field(default_factory=dict)  # per-stage latency: stt / llm / tts / total
    detected_language: str = ""  # language Sarvam reported for the input



... class VoiceAgent: transcribe() -> think() -> speak() -> respond() ...


## 4. Run it — a full spoken turn

We don't even need a microphone: `make_audio()` uses TTS to synthesize a question into a `.wav`, then `respond()` runs the whole pipeline on it. (Synthesizing our own test audio is also the seed of a Day 2 test — the **TTS→STT round-trip**.)

> **Live cell** — needs `SARVAM_API_KEY` + `GROQ_API_KEY` and `pip install -r ../requirements.txt`. Watch the `timings_ms`: that per-stage latency is exactly what Day 2's tests put budgets on.

In [4]:
import sys; sys.path.insert(0, ".")
from voice_agent import VoiceAgent

agent = VoiceAgent()

# 1) synthesize a spoken question (no mic needed)
question_wav = agent.make_audio(
    "What should I pack for a trip to Manali in December?", "audio/question.wav"
)

# 2) run the whole pipeline on it
turn = agent.respond(question_wav, out_path="audio/reply.wav")

print("Heard   :", turn.transcript)
print("Replied :", turn.reply_text)
print("Language:", turn.detected_language)
print("Audio   :", turn.audio_out_path)
print("Timings :", turn.timings_ms, "(ms)")

Heard   : That should be packed for a trip to Lanali in December.
Replied : You'll want to pack warm clothing for Lanali in December, it can get quite chilly.
Language: en-IN
Audio   : audio/reply.wav
Timings : {'stt': 476, 'llm': 404, 'llm_first_token': 385, 'tts': 512, 'total': 1393} (ms)


In [5]:
# Listen to both ends of the turn (in a notebook)
from IPython.display import Audio, display
print("Question (what we asked):");  display(Audio("audio/question.wav"))
print("Reply (what the agent said):"); display(Audio("audio/reply.wav"))

Question (what we asked):


Reply (what the agent said):


## 5. Make it conversational (a chatbot)

`respond()` is one-shot. **`VoiceChat`** wraps the same agent and **remembers the conversation**, so the brain has context turn to turn:

- `chat.say("...")` — you *type* a turn (skips STT); the reply is spoken.
- `chat.ask("clip.wav")` — you *speak* a turn (full STT → LLM → TTS).

Memory is itself a testable behaviour (Module 6): a follow-up like *"...and is it cold there at night?"* only makes sense if the bot kept context.

For a **live terminal chatbot**, run the standalone script instead:

```bash
python chat.py            # type turns, replies spoken aloud
python chat.py --mic      # speak your turns (needs: pip install sounddevice soundfile)
python chat.py --no-audio # text only, no playback
```

In [ ]:
# A short CONVERSATION (with memory). The second turn only makes sense if the
# bot remembers the first — that's the context-retention behaviour Day 2 tests.
from voice_agent import VoiceChat

chat = VoiceChat()
for line in [
    "What should I pack for a trip to Manali in December?",
    "And is it cold enough there for snow at night?",   # 'there' = only works WITH memory
]:
    turn = chat.say(line)
    print(f"you> {line}")
    print(f"bot> {turn.reply_text}   ({turn.timings_ms['total']} ms)\n")

In [ ]:
# Optional — an interactive typed chat right here in the notebook.
# Run this cell, type in the input box; 'quit' (or an empty line) to stop.
from voice_agent import VoiceChat

live = VoiceChat()
while True:
    user = input("you> ").strip()
    if user.lower() in {"quit", "exit", ""}:
        break
    t = live.say(user)
    print(f"bot> {t.reply_text}   ({t.timings_ms['total']} ms)  🔊 {t.audio_out_path}")
print("(chat ended)")

## 6. What we'll test in Day 2 (STT, TTS & the brain — with an LLM judge)

Today we **built** the agent. Day 2 **tests** it. This section just frames *what that suite will cover* — nothing runs here yet.

**Why an LLM judge?** You already saw exact string-matching won't work: the agent heard *"…Manali…"* as *"…Lanali…"*. Speech is noisy and replies are open-ended, so we grade **meaning, not exact strings** — with an **LLM-as-a-judge**, the same technique as Module 4's DeepEval `GEval` and Module 8's `llm-rubric`, now pointed at voice. One judging method, three stages:

| Stage | Day 2 test | What the judge is asked (plain English) |
|---|---|---|
| **STT (ears)** | Transcription fidelity | *"Does the transcript preserve the meaning of what was said? A changed place name or number is a fail."* |
| **TTS (mouth)** | Intelligibility — TTS→STT round-trip | *"Speak a line, transcribe it back: did the meaning (names, numbers) survive?"* |
| **LLM (brain)** | Reply quality | *"Is the reply relevant, safe, and speakable — 1–3 short sentences, no markdown?"* |

**What one of these will look like in Day 2** (illustrative — we won't run it today):

```python
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

reply_quality = GEval(
    name="Speakable & Relevant",
    criteria="The reply answers the question, is safe, and sounds natural aloud — "
             "1-3 short sentences, no markdown, lists, or symbols.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)
# Day 2: reply_quality.measure(LLMTestCase(input=transcript, actual_output=reply)) -> score + reason
```

**Also on the Day 2 menu (complementary, non-judge):**
- **Latency** — a budget on the `timings_ms` this agent already returns (Module 4 Day 5's `LatencyMetric`, now on voice).
- **WER** — word error rate with `jiwer` for the hard STT cases (numbers, names), alongside the fidelity judge.

Two things today's agent already hands Day 2 for free: **`timings_ms`** (→ latency checks) and **`make_audio()`** (→ the TTS→STT round-trip).

> **The through-line:** voice testing isn't a new philosophy — it's Module 4 Day 4's mindset (partition the inputs, test the edges, judge *meaning* not strings, treat latency as first-class) with **audio** as the input and **three stages** to cover. Day 2 builds exactly these.

## Try it yourself

1. **Break the ears.** In `make_audio()`, synthesize a question full of tricky tokens — a place name + a number + a Hindi-English mix (e.g. *"Mujhe Bengaluru mein 24 December ko kya pack karna chahiye?"*). Run `respond()`. Did the transcript come back clean? This is a preview of Day 2's STT/WER testing.
2. **Break the mouth's job.** Change the system prompt to *remove* the "no markdown, 1–3 sentences" rule and re-run. Does the reply now contain bullets or long text that sounds terrible aloud? That's the "un-speakable reply" failure — a real voice bug you'll write a test for.
3. **Feel the latency.** Print just `turn.timings_ms`. Which stage is slowest — ears, brain, or mouth? Swap `GROQ_MODEL` to `llama-3.3-70b-versatile` in `.env` and compare. This is the trade-off Day 2's latency budgets make explicit.

## Summary

- A **voice agent** is STT → LLM → TTS: three failure surfaces, and **latency** dominates the experience.
- We built a **local, file-in/file-out** agent (Sarvam ears+mouth, Groq brain) where every stage returns a value and is **timed** — built to be tested.
- The **brain** is tested exactly as before (promptfoo/DeepEval). What's **new** is STT accuracy, TTS intelligibility, end-to-end latency, turn-taking, and fallback paths — all still the Module 4 Day 4 mindset, with audio as the input.

**Next — Day 2:** implement these. WER on STT with accent/noise partitions, the TTS→STT round-trip, latency-budget assertions on `timings_ms`, and hard-negative fallback tests (silence, gibberish, out-of-scope) 